# 04 · Avaliação e registro

*Champion × challenger: só promove se for melhor*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## Por que um *quality gate*?

O pipeline re-treina **todo dia, sozinho**. Se um dia os dados vierem estranhos (sitemap
quebrado, bug na simulação), o modelo novo pode ser pior — e ninguém está olhando.

Regra: a versão nova (*challenger*) só recebe o alias **`@champion`** se o NDCG@10 dela
**superar** o do campeão atual. O app e a inferência batch sempre usam `@champion`,
nunca "a última versão". **Rollback** = apontar o alias para a versão anterior.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

In [ ]:
# MLflow: experimento por usuário + registro de modelos no Unity Catalog
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ME = w.current_user.me().user_name
EXPERIMENT = f"/Users/{ME}/news_recsys_{settings.schema_name}"
mlflow.set_experiment(EXPERIMENT)
mlflow.set_registry_uri("databricks-uc")
print("experimento:", EXPERIMENT)

### `registry.py`

In [ ]:
# src/news_recsys/registry.py  (somente leitura — o código de verdade está em src/)
"""Quality gate champion/challenger + helpers do Unity Catalog (imports lazy)."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class PromotionDecision:
    promote: bool
    reason: str
    candidate_value: float
    champion_value: float | None


def decide_promotion(
    candidate_metrics: dict[str, float],
    champion_metrics: dict[str, float] | None,
    metric: str = "ndcg_at_k",
    min_improvement: float = 0.0,
) -> PromotionDecision:
    """O desafiante só vira campeão se for MELHOR que o atual (com margem opcional).

    É a regra que impede um retreino ruim (dados estranhos, bug) de chegar ao app.
    """
    if metric not in candidate_metrics:
        raise KeyError(f"métrica {metric!r} ausente nas métricas do candidato")
    cand = float(candidate_metrics[metric])
    if champion_metrics is None or metric not in champion_metrics:
        return PromotionDecision(
            True, "não há campeão: o primeiro modelo vira champion", cand, None
        )
    champ = float(champion_metrics[metric])
    if cand > champ + min_improvement:
        return PromotionDecision(True, f"{metric} {cand:.4f} > campeão {champ:.4f}", cand, champ)
    return PromotionDecision(
        False,
        f"{metric} {cand:.4f} não supera campeão {champ:.4f} (+{min_improvement})",
        cand,
        champ,
    )


def champion_metrics(model_fqn: str, alias: str = "champion") -> dict[str, float] | None:
    """Métricas da run que gerou a versão ``@champion`` (None se ainda não existe)."""
    from mlflow import MlflowClient

    client = MlflowClient(registry_uri="databricks-uc")
    try:
        version = client.get_model_version_by_alias(model_fqn, alias)
    except Exception:  # noqa: BLE001 — modelo/alias inexistente
        return None
    if not version.run_id:
        return None
    return dict(client.get_run(version.run_id).data.metrics)


def set_champion(model_fqn: str, version: str | int, alias: str = "champion") -> Any:
    from mlflow import MlflowClient

    client = MlflowClient(registry_uri="databricks-uc")
    client.set_registered_model_alias(model_fqn, alias, str(version))
    return client.get_model_version_by_alias(model_fqn, alias)

### Execução: quem é o desafiante e quem é o campeão?

In [ ]:
import pandas as pd
from mlflow import MlflowClient

from news_recsys.registry import champion_metrics, decide_promotion, set_champion

client = MlflowClient(registry_uri="databricks-uc")
version = dbutils.jobs.taskValues.get(taskKey="treino", key="model_version", default="", debugValue="")
if not version:  # execução interativa: pega a versão mais recente
    version = max(client.search_model_versions(f"name='{settings.model_fqn}'"), key=lambda v: int(v.version)).version
candidate = client.get_model_version(settings.model_fqn, version)
candidate_metrics = client.get_run(candidate.run_id).data.metrics
current = champion_metrics(settings.model_fqn)

pd.DataFrame(
    {"desafiante (v" + str(version) + ")": candidate_metrics, "campeão atual": current or {}}
).loc[["ndcg_at_k", "precision_at_k", "recall_at_k", "hit_rate_at_k", "catalog_coverage", "cold_hit_rate_at_k"]].round(4)

### Execução: a decisão

In [ ]:
decision = decide_promotion(candidate_metrics, current, settings.promotion_metric, settings.min_improvement)
print("PROMOVER" if decision.promote else "MANTER CAMPEÃO", "—", decision.reason)

client.set_model_version_tag(settings.model_fqn, version, "gate_decision", "promoted" if decision.promote else "rejected")
client.set_model_version_tag(settings.model_fqn, version, "gate_reason", decision.reason)
if decision.promote:
    champ = set_champion(settings.model_fqn, version)
    print(f"@champion -> versão {champ.version}")

### Execução: histórico de métricas (tabela `model_metrics`)

Gravamos as métricas de **todos os candidatos** desta execução (lidas das runs filhas
via `mlflow.search_runs`). O app mostra essa tabela na aba "Modelo" e o notebook 06 usa
para acompanhar a tendência.

In [ ]:
from datetime import UTC, datetime

children = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{candidate.run_id}'")
winner = client.get_run(candidate.run_id).data.params.get("winner")
now = datetime.now(UTC).isoformat(timespec="seconds")
metric_names = ["ndcg_at_k", "precision_at_k", "recall_at_k", "hit_rate_at_k", "catalog_coverage", "cold_hit_rate_at_k"]
rows = [
    {
        "evaluated_at": now,
        "model_version": str(version),
        "candidate": run["tags.mlflow.runName"],
        "metric": m,
        "value": float(run[f"metrics.{m}"]),
        "promoted": bool(decision.promote and run["tags.mlflow.runName"] == winner),
    }
    for _, run in children.iterrows()
    for m in metric_names
]
spark.createDataFrame(pd.DataFrame(rows)).write.mode("append").saveAsTable(settings.table("model_metrics"))
spark.table(settings.table("model_metrics")).orderBy("evaluated_at", ascending=False).limit(10).toPandas()

### Versões e aliases no Unity Catalog

In [ ]:
pd.DataFrame(
    [
        {"versão": v.version, "aliases": list(v.aliases or []), "gate": (v.tags or {}).get("gate_decision"), "run_id": v.run_id}
        for v in client.search_model_versions(f"name='{settings.model_fqn}'")
    ]
).sort_values("versão", key=lambda s: s.astype(int), ascending=False)

### O que percebemos?

- A promoção é **automática e auditável** (tags `gate_decision`/`gate_reason` em cada versão).
- Consumidores dependem do **alias**, não do número da versão → deploy e rollback sem mexer no app.
- Rode o Job duas vezes com o mesmo `seed`: a segunda versão empata e **não** é promovida.

**Próximo:** `05_inferencia_batch`.